In [ ]:
# Setup
import asyncio
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("✓ Environment ready")

## 1. Document Ingestion

First, let's ingest some documents into the system.

In [ ]:
from src.ingestion.pipeline import IngestionPipeline

async def ingest_documents():
    pipeline = IngestionPipeline(enable_kg=True)
    await pipeline.initialize()
    
    # Ingest from big_tech_docs directory
    docs_dir = Path("../big_tech_docs")
    if docs_dir.exists():
        stats = await pipeline.ingest_directory(docs_dir)
        print(f"\n✓ Ingestion Complete!")
        print(f"  Documents: {stats['documents_processed']}")
        print(f"  Chunks: {stats['chunks_created']}")
        print(f"  KG Entities: {stats['knowledge_graph']['entities_created']}")
        print(f"  Duration: {stats['duration_seconds']:.2f}s")
    else:
        print(f"Directory not found: {docs_dir}")

# Run ingestion (only needed once)
# await ingest_documents()
print("Ingestion cell ready (uncomment to run)")

## 2. Query with Different RAG Variants

Let's test a query using all three RAG approaches.

In [ ]:
from src.rag_variants.naive_rag import NaiveRAG
from src.rag_variants.kg_rag import KnowledgeGraphRAG
from src.rag_variants.hybrid_rag import HybridRAG

# Define test query
query = "What is the relationship between Microsoft and OpenAI?"

async def compare_rag_variants(question: str):
    # Initialize all RAG systems
    naive = NaiveRAG(top_k=5)
    kg = KnowledgeGraphRAG(top_k=5)
    hybrid = HybridRAG(top_k=5, fusion_strategy="rrf")
    
    await naive.initialize()
    await kg.initialize()
    await hybrid.initialize()
    
    print(f"\nQuestion: {question}\n")
    print("=" * 80)
    
    # Run Naive RAG
    print("\n1. NAIVE RAG (Vector Similarity)")
    print("-" * 80)
    naive_result = await naive.generate(question)
    print(f"Answer: {naive_result['answer']}")
    print(f"Latency: {naive_result['latency_ms']:.2f}ms")
    print(f"Chunks: {len(naive_result['retrieved_chunks'])}")
    
    # Run KG RAG
    print("\n2. KNOWLEDGE GRAPH RAG")
    print("-" * 80)
    kg_result = await kg.generate(question)
    print(f"Answer: {kg_result['answer']}")
    print(f"Latency: {kg_result['latency_ms']:.2f}ms")
    print(f"Chunks: {len(kg_result['retrieved_chunks'])}")
    
    # Run Hybrid RAG
    print("\n3. HYBRID RAG (Vector + KG)")
    print("-" * 80)
    hybrid_result = await hybrid.generate(question)
    print(f"Answer: {hybrid_result['answer']}")
    print(f"Latency: {hybrid_result['latency_ms']:.2f}ms")
    print(f"Chunks: {len(hybrid_result['retrieved_chunks'])}")
    print(f"Fusion: {hybrid_result['fusion_strategy']}")

# Run comparison
await compare_rag_variants(query)

## 3. Run Comprehensive Evaluation

Evaluate all three variants on a benchmark dataset with LLM-based scoring.

In [ ]:
from src.evaluation.orchestrator import EvaluationOrchestrator
import json

async def run_evaluation():
    orchestrator = EvaluationOrchestrator()
    await orchestrator.initialize()
    
    # Run evaluation on 10 questions
    summary = await orchestrator.run_evaluation(
        dataset_name="hotpotqa",
        num_questions=10,
        output_dir="../benchmarks/results"
    )
    
    print("\n" + "=" * 80)
    print("EVALUATION SUMMARY")
    print("=" * 80)
    print(json.dumps(summary, indent=2))
    
    return summary

# Run evaluation (takes a few minutes)
# summary = await run_evaluation()
print("Evaluation cell ready (uncomment to run)")

## 4. Analyze Results

Visualize the comparison between RAG variants.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_comparison(summary):
    """Plot comparison of RAG variants."""
    variants = ['Naive RAG', 'KG RAG', 'Hybrid RAG']
    dimensions = ['Correctness', 'Completeness', 'Relevance', 'Faithfulness', 'Clarity']
    
    # Extract scores
    naive_scores = list(summary['naive_rag']['avg_scores'].values())
    kg_scores = list(summary['kg_rag']['avg_scores'].values())
    hybrid_scores = list(summary['hybrid_rag']['avg_scores'].values())
    
    # Create grouped bar chart
    x = np.arange(len(dimensions))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width, naive_scores, width, label='Naive RAG')
    ax.bar(x, kg_scores, width, label='KG RAG')
    ax.bar(x + width, hybrid_scores, width, label='Hybrid RAG')
    
    ax.set_xlabel('Evaluation Dimension')
    ax.set_ylabel('Score (1-5)')
    ax.set_title('RAG Variant Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(dimensions)
    ax.legend()
    ax.set_ylim(0, 5)
    
    plt.tight_layout()
    plt.show()

# Plot results (requires summary from evaluation)
# plot_comparison(summary)
print("Plotting cell ready")

## Next Steps

- Load your own documents
- Create custom benchmark questions
- Experiment with different chunk sizes and overlap
- Try different fusion strategies for Hybrid RAG
- Analyze detailed evaluation results

## CLI Commands

You can also use the CLI for these operations:

```bash
# Ingest documents
python cli.py ingest ./big_tech_docs

# Run evaluation
python cli.py evaluate --num-questions 50

# Query the system
python cli.py query "Your question?" --variant hybrid

# Check status
python cli.py status
```